In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from prophet import Prophet
from xgboost import XGBRegressor, XGBClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, 
    mean_absolute_percentage_error, 
    mean_squared_error,
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    roc_auc_score, 
    roc_curve
)
import shap
import lime
import lime.lime_tabular
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [15]:
import kagglehub
path = kagglehub.dataset_download('robikscube/hourly-energy-consumption')

print('Data source import complete.')


Data source import complete.


dataları birləşdiririk


In [16]:
import os

files =os.listdir(path)
print(files)

['AEP_hourly.csv', 'COMED_hourly.csv', 'DAYTON_hourly.csv', 'DEOK_hourly.csv', 'DOM_hourly.csv', 'DUQ_hourly.csv', 'EKPC_hourly.csv', 'est_hourly.paruqet', 'FE_hourly.csv', 'NI_hourly.csv', 'PJME_hourly.csv', 'PJMW_hourly.csv', 'pjm_hourly_est.csv', 'PJM_Load_hourly.csv']


In [17]:
all_dfs= []

for file in os.listdir(path):
    if file.endswith(".csv") and file != "pjm_hourly_est.csv":
        df=pd.read_csv(os.path.join(path,file))
        region =df.columns[1].replace("_MW","")
        df.columns =["Datetime","Load"]
        df['region'] =region
        all_dfs.append(df)

data =pd.concat(all_dfs, ignore_index=True)
data["Datetime"] = pd.to_datetime(data["Datetime"])
data = data.set_index("Datetime").sort_index()


print(data.head())
print(data["region"].unique())
print(data.shape)

                        Load    region
Datetime                              
1998-04-01 01:00:00  22259.0  PJM_Load
1998-04-01 02:00:00  21244.0  PJM_Load
1998-04-01 03:00:00  20651.0  PJM_Load
1998-04-01 04:00:00  20421.0  PJM_Load
1998-04-01 05:00:00  20713.0  PJM_Load
<StringArray>
['PJM_Load',     'PJME',     'PJMW',       'NI',      'AEP',   'DAYTON',
      'DUQ',      'DOM',    'COMED',       'FE',     'DEOK',     'EKPC']
Length: 12, dtype: str
(1090167, 2)


In [18]:
data.head()

,Load,region
Datetime,,
1998-04-01 01:00:00,22259.0,PJM_Load
1998-04-01 02:00:00,21244.0,PJM_Load
1998-04-01 03:00:00,20651.0,PJM_Load
1998-04-01 04:00:00,20421.0,PJM_Load
1998-04-01 05:00:00,20713.0,PJM_Load


yeni sütunlar əlave edirik

In [19]:
data['year']= data.index.year
data['month']= data.index.month
data['day']= data.index.day
data['hour']= data.index.hour
data['dayofweek']= data.index.dayofweek
data['weekday_name']= data.index.day_name()
data['month_name']= data.index.month_name()
data['quarter']= data.index.quarter

data['is_weekend']= data['dayofweek'].isin([5,6]).astype(int)
data['dayofyear'] = data.index.dayofyear
data['weekofyear']=data.index.isocalendar().week.astype(int)
data.head()

,Load,region,year,month,day,hour,dayofweek,weekday_name,month_name,quarter,is_weekend,dayofyear,weekofyear
Datetime,,,,,,,,,,,,,
1998-04-01 01:00:00,22259.0,PJM_Load,1998,4,1,1,2,Wednesday,April,2,0,91,14
1998-04-01 02:00:00,21244.0,PJM_Load,1998,4,1,2,2,Wednesday,April,2,0,91,14
1998-04-01 03:00:00,20651.0,PJM_Load,1998,4,1,3,2,Wednesday,April,2,0,91,14
1998-04-01 04:00:00,20421.0,PJM_Load,1998,4,1,4,2,Wednesday,April,2,0,91,14
1998-04-01 05:00:00,20713.0,PJM_Load,1998,4,1,5,2,Wednesday,April,2,0,91,14


In [20]:
data.dtypes

Load            float64
region              str
year              int32
month             int32
day               int32
hour              int32
dayofweek         int32
weekday_name        str
month_name          str
quarter           int32
is_weekend        int64
dayofyear         int32
weekofyear        int64
dtype: object

In [21]:
data = pd.get_dummies(data, columns=['region', 'weekday_name', 'month_name'], drop_first=False)
 

In [22]:

# 1. Datanı hazırla (2016-dan sonra)
data_fast = data[data.index > '2016-01-01'].copy()

# 2. Feature Engineering - Lag + Rolling + External Regressors
data_fast['lag_48'] = data_fast['Load'].shift(48)
data_fast['lag_72'] = data_fast['Load'].shift(72)
data_fast['lag_168'] = data_fast['Load'].shift(168)

data_fast['rolling_mean_24'] = data_fast['Load'].rolling(24).mean()
data_fast['rolling_std_24'] = data_fast['Load'].rolling(24).std()
data_fast['rolling_max_24'] = data_fast['Load'].rolling(24).max()
data_fast['rolling_min_24'] = data_fast['Load'].rolling(24).min()
# External Regressors (əgər datada varsa)
# Temperature, Humidity kimi sütunlar varsa:
if 'Temperature' in data_fast.columns:
    data_fast['temp_lag_1'] = data_fast['Temperature'].shift(1)
if 'Humidity' in data_fast.columns:
    data_fast['humidity_lag_1'] = data_fast['Humidity'].shift(1)

data_fast.dropna(inplace=True)

# 3. Prophet modeli (external regressors ilə)
df_p = data_fast.reset_index()[['Datetime', 'Load']].copy()
df_p.columns = ['ds', 'y']

# External regressors əlavə et
if 'Temperature' in data_fast.columns:
    df_p['temperature'] = data_fast['Temperature'].values

model_p = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True, uncertainty_samples=0)

# External regressors register et
if 'temperature' in df_p.columns:
    model_p.add_regressor('temperature')

model_p.fit(df_p)

forecast = model_p.predict(df_p[['ds', 'temperature']] if 'temperature' in df_p.columns else df_p[['ds']])
forecast.index = data_fast.index
data_fast['prophet_pred'] = forecast['yhat'].values
data_fast['residuals'] = data_fast['Load'] - data_fast['prophet_pred']

# 4. XGBoost üçün xüsusiyyətləri hazırla
cols_to_drop = ['Load', 'residuals', 'prophet_pred', 'region', 'weekday_name', 'month_name', 'Datetime']
X = data_fast.drop(columns=[c for c in cols_to_drop if c in data_fast.columns])
X = X.astype(float)
y_res = data_fast['residuals']

# 5. Train/Test split
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_res_train, y_res_test = y_res.iloc[:split], y_res.iloc[split:]

# 6. XGBoost eğit (Hybrid approach)
model_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model_xgb.fit(X_train, y_res_train)

# 7. Hybrid Proqnoz
xgb_pred = model_xgb.predict(X_test)
final_forecast = data_fast['prophet_pred'].iloc[split:].values + xgb_pred

# 8. Metrikalar
y_true = data_fast['Load'].iloc[split:].values

mae = mean_absolute_error(y_true, final_forecast)
rmse = np.sqrt(mean_squared_error(y_true, final_forecast))
mape = mean_absolute_percentage_error(y_true, final_forecast)
r2 = 1 - (np.sum((y_true - final_forecast)**2) / np.sum((y_true - y_true.mean())**2))
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape*100:.2f}%")
print(f"R²:   {r2:.4f}")


15:01:16 - cmdstanpy - INFO - Chain [1] start processing
15:01:40 - cmdstanpy - INFO - Chain [1] done processing


MAE:  387.98
RMSE: 596.45
MAPE: 7.10%
R²:   0.9956
